In [ ]:
import socket
import time
import random
import sys

from scapy.all import IP, TCP, send, Raw
from datetime import datetime

SERVER_IP_MAPPING = {

    "10.0.2.10": 1,
    "10.0.2.11": 2,
    "10.0.2.12": 3,
    "10.0.2.13": 4
}

IP_RANGES = {

    1: (1, 50),
    2: (65, 115),
    3: (130, 180),
    4: (195, 245)
}

def get_local_ip_address():

    try:

        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.connect(("10.0.2.1", 80))
        ip_address = sock.getsockname()[0]
        sock.close()

        return ip_address
    
    except Exception:

        return None

def generate_spoofed_source_ip(server_id, base_ip_prefix="10.0.2"):

    min_byte, max_byte = IP_RANGES[server_id]
    last_octet = random.randint(min_byte, max_byte)

    return f"{base_ip_prefix}.{last_octet}"

def send_spoofed_http_request(source_ip, target_ip, target_port, user_agent_list):

    source_port = random.randint(1024, 65535)
    initial_sequence = random.randint(100000, 4000000000)

    syn_packet = IP(src=source_ip, dst=target_ip, ttl=64) / \
                 TCP(sport=source_port, dport=target_port,
                     flags="S", window=64240, seq=initial_sequence)
    
    send(syn_packet, verbose=False)

    time.sleep(random.uniform(0.001, 0.005))

    http_payload = (

        f"GET /index.html HTTP/1.1\r\n"
        f"Host: {target_ip}\r\n"
        f"User-Agent: {random.choice(user_agent_list)}\r\n"
        f"Accept: text/html\r\n\r\n"
    )

    push_ack_packet = IP(src=source_ip, dst=target_ip, ttl=64) / \
                      TCP(sport=source_port, dport=target_port,
                          flags="PA", window=64240,
                          seq=initial_sequence + 1,
                          ack=random.randint(100000, 4000000000)) / \
                      Raw(load=http_payload)
    
    send(push_ack_packet, verbose=False)

def generate_legitimate_traffic(server_id, target_ip="10.0.3.10", target_port=80,
                                normal_burst_range=(3, 8), peak_burst_range=(30, 50)):
    
    print(f"Avvio generatore di traffico legittimo (Server {server_id})")
    print(f"Intervallo IP sorgente: 10.0.2.{IP_RANGES[server_id][0]}-{IP_RANGES[server_id][1]}")

    total_packets = 0

    user_agent_list = [

        "Mozilla/5.0 (Windows NT 10.0)",
        "iPhone; CPU OS 14_4",
        "Go-http-client/1.1"
    ]

    start_time = time.time()

    if server_id in (1, 2, 4):

        peak_count = 0
        last_peak_time = start_time

        if server_id == 1:

            peak_interval = 600
            initial_threshold = 10

        elif server_id == 2:

            peak_interval = 720
            initial_threshold = 30

        elif server_id == 4:

            peak_interval = 900
            initial_threshold = 60

    if server_id == 3:

        last_peak_time = time.time()
        next_peak_interval = random.uniform(15, 45)

    def execute_burst(request_count, label=""):

        nonlocal total_packets

        if label:

            print(f"[{label}] Server {server_id}: invio {request_count} richieste in picco")

        else:

            print(f"[{datetime.now().strftime('%H:%M:%S')}] Burst normale di {request_count} richieste")

        for _ in range(request_count):

            source_ip = generate_spoofed_source_ip(server_id)

            try:

                send_spoofed_http_request(source_ip, target_ip, target_port, user_agent_list)

            except Exception as error:

                print(f"[ERRORE] Invio pacchetto: {error}")

            total_packets += 1
            sleep_interval = random.uniform(0.1, 0.5) if not label else random.uniform(0.05, 0.2)
            time.sleep(sleep_interval)

    try:
        while True:

            execute_burst(random.randint(*normal_burst_range))

            elapsed_time = time.time() - start_time

            if server_id == 1 and peak_count < 3:

                if peak_count == 0 and elapsed_time >= initial_threshold:

                    execute_burst(random.randint(*peak_burst_range), label="PICCO")
                    peak_count += 1
                    last_peak_time = time.time()

                elif peak_count > 0 and (time.time() - last_peak_time) >= peak_interval:

                    execute_burst(random.randint(*peak_burst_range), label="PICCO")
                    peak_count += 1
                    last_peak_time = time.time()

            elif server_id == 2 and peak_count < 3:

                if peak_count == 0 and elapsed_time >= initial_threshold:

                    execute_burst(random.randint(*peak_burst_range), label="PICCO")
                    peak_count += 1
                    last_peak_time = time.time()

                elif peak_count > 0 and (time.time() - last_peak_time) >= peak_interval:

                    execute_burst(random.randint(*peak_burst_range), label="PICCO")
                    peak_count += 1
                    last_peak_time = time.time()

            elif server_id == 3 and (time.time() - last_peak_time) >= next_peak_interval:

                execute_burst(random.randint(*peak_burst_range), label="PICCO")
                last_peak_time = time.time()
                next_peak_interval = random.uniform(15, 45)

            elif server_id == 4 and peak_count < 3:

                if peak_count == 0 and elapsed_time >= initial_threshold:

                    execute_burst(random.randint(*peak_burst_range), label="PICCO")
                    peak_count += 1
                    last_peak_time = time.time()

                elif peak_count > 0 and (time.time() - last_peak_time) >= peak_interval:

                    execute_burst(random.randint(*peak_burst_range), label="PICCO")
                    peak_count += 1
                    last_peak_time = time.time()

            pause_duration = random.uniform(10, 30)
            print(f"Pausa di {pause_duration:.1f}s ... (Totale pacchetti: {total_packets})")
            time.sleep(pause_duration)

    except KeyboardInterrupt:

        print(f"\n[INFO] Generazione terminata. Pacchetti inviati: {total_packets}")
        sys.exit(0)

if __name__ == "__main__":

    real_ip = get_local_ip_address()

    if real_ip is None or real_ip not in SERVER_IP_MAPPING:

        print(f"ERRORE: Indirizzo IP {real_ip} non riconosciuto.")
        sys.exit(1)

    server_id = SERVER_IP_MAPPING[real_ip]
    print(f"[INFO] Riconosciuto server {server_id} con IP {real_ip}")
    generate_legitimate_traffic(server_id)